# Google Gemini

In [ ]:
from langchain_google_genai import ChatGoogleGenerativeAI
from langchain_core.prompts import ChatPromptTemplate
import os
from dotenv import load_dotenv

# Load environment variables
load_dotenv()

# Initialize the model
llm = ChatGoogleGenerativeAI(
    model="gemini-2.5-flash",  # Use latest model: gemini-2.0-flash, gemini-1.5-pro, etc.
    temperature=0.7
)

# Create a prompt template
prompt = ChatPromptTemplate.from_messages([
    ("system", "You are a helpful assistant that explains concepts clearly."),
    ("human", "Explain {topic} in simple terms")
])

# Create a chain using the new syntax (LLMChain is deprecated)
chain = prompt | llm

# Run the chain
result = chain.invoke({"topic": "quantum computing"})
print(result.content)

# Open Source Models with Ollama

In [ ]:
from langchain_ollama import OllamaLLM
from langchain_core.prompts import ChatPromptTemplate

In [ ]:
# Initialize different models
llama = OllamaLLM(model="gemma:2b")

In [ ]:
# Create a prompt template
prompt = ChatPromptTemplate.from_messages([
    ("system", "You are a helpful assistant that explains concepts clearly."),
    ("human", "Explain {topic} in simple terms")
])

# Create a chain using the new syntax (LLMChain is deprecated)
chain = prompt | llama

# Run the chain
result = chain.invoke({"topic": "quantum computing"})
print(result)

# Adding Memory to Your Application

## Gemini

In [1]:
from langchain_google_genai import ChatGoogleGenerativeAI
from langchain_core.prompts import ChatPromptTemplate, MessagesPlaceholder
from langchain_community.chat_message_histories import ChatMessageHistory
from langchain_core.runnables.history import RunnableWithMessageHistory
from dotenv import load_dotenv

load_dotenv()

# Initialize the model
llm = ChatGoogleGenerativeAI(
    model="gemini-2.5-flash",
    temperature=0.7
)

# Conversation prompt using MessagesPlaceholder (explicit, recommended)
conversation_prompt = ChatPromptTemplate.from_messages([
    ("system", "You are a helpful assistant. Use the conversation history to provide context-aware responses."),
    MessagesPlaceholder(variable_name="chat_history"),
    ("human", "{input}")
])

# Build the base chain
chain = conversation_prompt | llm

# In-memory session store
session_store: dict[str, ChatMessageHistory] = {}

def get_session_history(session_id: str) -> ChatMessageHistory:
    if session_id not in session_store:
        session_store[session_id] = ChatMessageHistory()
    return session_store[session_id]

# Wrap chain with automatic history management
conversation_chain = RunnableWithMessageHistory(
    chain,
    get_session_history,
    input_messages_key="input",
    history_messages_key="chat_history"
)

# Helper — session_id lets you run multiple independent conversations
def chat(message: str, session_id: str = "default") -> str:
    response = conversation_chain.invoke(
        {"input": message},
        config={"configurable": {"session_id": session_id}}
    )
    return response.content

# Have a conversation
print(chat("Hi, my name is Alice"))
print(chat("What's my name?"))
print(chat("Tell me a joke about my name"))

C:\Users\NGHIANGUYEN\miniconda3\envs\agentic\Lib\site-packages\IPython\core\interactiveshell.py:3748: LangChainDeprecationWarning: RunnableWithMessageHistory is deprecated. Use LangGraph's built-in persistence instead.
  exec(code_obj, self.user_global_ns, self.user_ns)


Hello Alice! It's nice to meet you. How can I assist you today?
Your name is Alice.
Okay, Alice, here's one for you:

Why did Alice bring a ladder to the library?
Because she heard they had a lot of "tall" tales, and she wanted to make sure she could *reach* them all, just in case she fell "down the rabbit hole" of a good book!


In [4]:
print(chat("What have we discussed?"))

We've discussed:

1.  **Your name:** You told me your name is Alice.
2.  **My greeting:** I greeted you and offered assistance.
3.  **A joke:** I told you a joke that played on your name, Alice.


In [5]:
print(chat("What have we discussed?", session_id="randomid"))

We still haven't discussed anything. This is our first interaction, so there's no previous conversation to refer to!
